In [2]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

## 1. Load the cleaned dataset

In [6]:

file_name = "cleaned_synthetic_users_20260831.csv"

df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1500, 16)


,sessions_count,avg_engagement_time_sec,pageviews_per_session,compare_ratio,specials_ratio,mylists_ratio,category_browse_ratio,product_detail_ratio,unique_pages_visited,browsing_entropy,days_since_first_visit,days_since_last_activity,user_pseudo_id,reporting_window_start,reporting_window_end,is_included_in_run
0,1,15.39,2.67,0.0000,0.0000,0.0000,0.0382,0.0255,3,0.0112,10,10,syn_20260831_000001,2026-08-17,2026-08-30,1
1,5,194.98,9.85,0.0618,0.0694,0.2750,0.1673,0.0697,19,0.7500,14,1,syn_20260831_000002,2026-08-17,2026-08-30,1
2,3,163.32,5.57,0.0982,0.0104,0.1000,0.1514,0.3000,8,0.4323,9,6,syn_20260831_000003,2026-08-17,2026-08-30,1
3,4,97.80,6.74,0.3371,0.3073,0.0233,0.0500,0.0219,14,0.6781,8,0,syn_20260831_000004,2026-08-17,2026-08-30,1
4,5,94.89,6.99,0.1500,0.3113,0.0392,0.1342,0.0788,15,0.8408,8,4,syn_20260831_000005,2026-08-17,2026-08-30,1


## 2. Check the dataset before scaling

The previous task already handled the main cleaning.  
I am just doing a few checks here before preparing the clustering features.


In [7]:
print("Missing values:")
display(df.isnull().sum())

print("\nDuplicate user IDs:", df["user_pseudo_id"].duplicated().sum())

print("\nIncluded rows:")
print(df["is_included_in_run"].value_counts(dropna=False))

Missing values:


sessions_count              0
avg_engagement_time_sec     0
pageviews_per_session       0
compare_ratio               0
specials_ratio              0
mylists_ratio               0
category_browse_ratio       0
product_detail_ratio        0
unique_pages_visited        0
browsing_entropy            0
days_since_first_visit      0
days_since_last_activity    0
user_pseudo_id              0
reporting_window_start      0
reporting_window_end        0
is_included_in_run          0
dtype: int64


Duplicate user IDs: 0

Included rows:
is_included_in_run
1    1500
Name: count, dtype: int64


## 3. Keep the rows used for clustering

In [8]:
# Keep only rows marked as suitable for the clustering run
df_clustering = df[df["is_included_in_run"] == 1].copy()

print("Rows kept for clustering:", len(df_clustering))

Rows kept for clustering: 1500


## 4. Select the behavioural features

The ID and reporting dates are useful for tracking users, but they should not be used as clustering inputs.

Only the behavioural variables below are scaled and passed to the clustering model.


In [9]:
features = [
    "sessions_count",
    "avg_engagement_time_sec",
    "pageviews_per_session",
    "compare_ratio",
    "specials_ratio",
    "mylists_ratio",
    "category_browse_ratio",
    "product_detail_ratio",
    "unique_pages_visited",
    "browsing_entropy",
    "days_since_first_visit",
    "days_since_last_activity"
]

X = df_clustering[features].copy()

print("Clustering feature shape:", X.shape)
X.head()

Clustering feature shape: (1500, 12)


,sessions_count,avg_engagement_time_sec,pageviews_per_session,compare_ratio,specials_ratio,mylists_ratio,category_browse_ratio,product_detail_ratio,unique_pages_visited,browsing_entropy,days_since_first_visit,days_since_last_activity
0,1,15.39,2.67,0.0000,0.0000,0.0000,0.0382,0.0255,3,0.0112,10,10
1,5,194.98,9.85,0.0618,0.0694,0.2750,0.1673,0.0697,19,0.7500,14,1
2,3,163.32,5.57,0.0982,0.0104,0.1000,0.1514,0.3000,8,0.4323,9,6
3,4,97.80,6.74,0.3371,0.3073,0.0233,0.0500,0.0219,14,0.6781,8,0
4,5,94.89,6.99,0.1500,0.3113,0.0392,0.1342,0.0788,15,0.8408,8,4


## 5. Check the original feature ranges

The variables are measured on different scales.  
For example, engagement time is measured in seconds while several behavioural features are ratios.

This is why scaling is needed before distance-based clustering.


In [10]:
feature_ranges = pd.DataFrame({
    "min": X.min(),
    "max": X.max()
})

feature_ranges

,min,max
sessions_count,1.0,8.00
avg_engagement_time_sec,5.0,300.00
pageviews_per_session,2.0,10.00
compare_ratio,0.0,0.35
specials_ratio,0.0,0.35
mylists_ratio,0.0,0.30
category_browse_ratio,0.0,0.25
product_detail_ratio,0.0,0.30
unique_pages_visited,2.0,20.00
browsing_entropy,0.0,0.85


## 6. Apply Min-Max normalisation

Min-Max scaling transforms each feature to the range 0 to 1.

This keeps the features on a comparable scale so that variables with larger original values do not dominate the clustering distance calculations.


In [11]:
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

scaled_columns = ["mm_" + col for col in features]

scaled_df = pd.DataFrame(
    X_scaled,
    columns=scaled_columns,
    index=df_clustering.index
)

scaled_df.head()

,mm_sessions_count,mm_avg_engagement_time_sec,mm_pageviews_per_session,mm_compare_ratio,mm_specials_ratio,mm_mylists_ratio,mm_category_browse_ratio,mm_product_detail_ratio,mm_unique_pages_visited,mm_browsing_entropy,mm_days_since_first_visit,mm_days_since_last_activity
0,0.000000,0.035220,0.08375,0.000000,0.000000,0.000000,0.1528,0.085000,0.055556,0.013176,0.692308,1.0
1,0.571429,0.644000,0.98125,0.176571,0.198286,0.916667,0.6692,0.232333,0.944444,0.882353,1.000000,0.1
2,0.285714,0.536678,0.44625,0.280571,0.029714,0.333333,0.6056,1.000000,0.333333,0.508588,0.615385,0.6
3,0.428571,0.314576,0.59250,0.963143,0.878000,0.077667,0.2000,0.073000,0.666667,0.797765,0.538462,0.0
4,0.571429,0.304712,0.62375,0.428571,0.889429,0.130667,0.5368,0.262667,0.722222,0.989176,0.538462,0.4


## 7. Create the clustering-ready dataset

The user ID and reporting window are added back for traceability, but they are not part of the clustering model features.


In [12]:
clustering_ready = pd.concat(
    [
        df_clustering[["user_pseudo_id"]],
        scaled_df,
        df_clustering[["reporting_window_start", "reporting_window_end"]]
    ],
    axis=1
)

print("Final clustering-ready shape:", clustering_ready.shape)
clustering_ready.head()

Final clustering-ready shape: (1500, 15)


,user_pseudo_id,mm_sessions_count,mm_avg_engagement_time_sec,mm_pageviews_per_session,mm_compare_ratio,mm_specials_ratio,mm_mylists_ratio,mm_category_browse_ratio,mm_product_detail_ratio,mm_unique_pages_visited,mm_browsing_entropy,mm_days_since_first_visit,mm_days_since_last_activity,reporting_window_start,reporting_window_end
0,syn_20260831_000001,0.000000,0.035220,0.08375,0.000000,0.000000,0.000000,0.1528,0.085000,0.055556,0.013176,0.692308,1.0,2026-08-17,2026-08-30
1,syn_20260831_000002,0.571429,0.644000,0.98125,0.176571,0.198286,0.916667,0.6692,0.232333,0.944444,0.882353,1.000000,0.1,2026-08-17,2026-08-30
2,syn_20260831_000003,0.285714,0.536678,0.44625,0.280571,0.029714,0.333333,0.6056,1.000000,0.333333,0.508588,0.615385,0.6,2026-08-17,2026-08-30
3,syn_20260831_000004,0.428571,0.314576,0.59250,0.963143,0.878000,0.077667,0.2000,0.073000,0.666667,0.797765,0.538462,0.0,2026-08-17,2026-08-30
4,syn_20260831_000005,0.571429,0.304712,0.62375,0.428571,0.889429,0.130667,0.5368,0.262667,0.722222,0.989176,0.538462,0.4,2026-08-17,2026-08-30


## 8. Verify the scaled values

In [13]:
print("Minimum scaled value:", scaled_df.min().min())
print("Maximum scaled value:", scaled_df.max().max())

display(
    pd.DataFrame({
        "scaled_min": scaled_df.min(),
        "scaled_max": scaled_df.max()
    })
)

Minimum scaled value: 0.0
Maximum scaled value: 1.0000000000000002


,scaled_min,scaled_max
mm_sessions_count,0.0,1.0
mm_avg_engagement_time_sec,0.0,1.0
mm_pageviews_per_session,0.0,1.0
mm_compare_ratio,0.0,1.0
mm_specials_ratio,0.0,1.0
mm_mylists_ratio,0.0,1.0
mm_category_browse_ratio,0.0,1.0
mm_product_detail_ratio,0.0,1.0
mm_unique_pages_visited,0.0,1.0
mm_browsing_entropy,0.0,1.0


## 9. Save the outputs

Two files are saved:

1. **Clustering-ready dataset** - includes the user ID, scaled behavioural features and reporting dates.
2. **Model matrix** - contains only the scaled numeric features and can be passed directly to K-means or DBSCAN.


In [16]:
clustering_ready.to_csv(
    "discountmate_clustering_ready_minmax_20260831.csv",
    index=False
)

scaled_df.reset_index(drop=True).to_csv(
    "discountmate_clustering_model_matrix_minmax_20260831.csv",
    index=False
)